In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

credential_key=os.getenv("credential_key")
send_system_name=os.getenv("send_system_name")
model=os.getenv("model")
api_base_url=os.getenv("api_base_url")
user_id=os.getenv("user_id")

rag_credential_key = os.getenv("rag_credential_key")

문서 준비 : raw_text 형태로 - parsing.py의 결과를 이용해서 `list[str]` 형태로 만들어 봅시다!

In [ ]:
import json, requests

headers = {
    'Content=Type': 'application/json',
    'x-dep-ticket': rag_credential_key,
    'api-key': rag_api_key
}

payload = {
    'index_name': index_name,
    'data': {
        'doc_id': "키키테크",
        "title": "키키테크",
        "content": data
    },
    "chunk_factor": {
        "logic": "fixed_size",
        "chunk_size": 120,
        "chunk_overlap": 60,
        "separator": " "
    }
}

url = search_url + "/dsllmrag/elastic/v2/insert-doc"
response = requests.post(url, headers=headers, data=json.dumps(payload))

print(response)
print(response.text)

관련 문서 찾아오기

In [ ]:
user_prompt = "키키테크 임직원 수는?"

fields = {
    "index_name": index_name,
    "permission_groups": ['rag-public'],
    "query_text": user_prompt,
    "num_result_doc": 5,
    "fields_exclude": ["v_merge_title_content"]
}

json_data = json.dumps(fields)

# /dsllmrag/elastic/v2/retrieve-bm25, knn, rrf, cc
response = requests.request("POST", search_url + "/dsllmrag/elastic/v2/retrieve-bm25", data=json_data, headers=headers)

result = response.json()
rag_context = result['hits']['hits'][0]['_source']['merge_title_content']
print(rag_context)

문서 삭제

In [ ]:
fields = {
    "index_name": index_name,
    "permission_groups": ['rag-public'],
    "doc_id": "키키테크"
}

json_data = json.dump(fields)

response = requests.request("POST", search_url + "/ds_llm_rag/elastic/v2/delete-doc", data=json_data, headers=headers)

print(response)

함수 및 툴로 정리하고 에이전트에서 사용해보기